# Evaluación dirigida a días festivos

**Añadir al notebook 03 como sección 7.2.**

El fichero `error_por_tipo_dia.csv` muestra la columna *Festivo* vacía: ninguno
de los 14 orígenes de la evaluación general cayó en un día festivo. El periodo
cubierto —del 23 de octubre al 27 de diciembre de 2024, con orígenes cada cinco
días— no coincidió con ninguno.

Esto deja sin contrastar el hallazgo central del trabajo. El análisis de
anomalías del notebook 01 concluyó que cerca del 70 % de las desviaciones
significativas se concentraba en festivos, y sobre esa base se incorporaron las
variables `festivo`, `vispera_festivo` y `post_festivo`. Conviene por tanto
verificar explícitamente que dichas variables cumplen su función.

Se construye para ello un conjunto de evaluación **dirigido**: orígenes
seleccionados de modo que el día predicho sea festivo.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 7.2.1 — Orígenes situados en días festivos
# ═══════════════════════════════════════════════════════════════════════════
import holidays as _holidays

ANIOS = range(y.index.year.min(), y.index.year.max() + 1)
FESTIVOS = pd.to_datetime(sorted(_holidays.Spain(years=ANIOS, subdiv="MD").keys()))


def ventanas_festivos(indice, festivos, horizonte, min_train=24*365, n_max=10):
    """Genera orígenes cuyo día predicho completo es festivo."""
    ventanas = []
    for fecha in sorted(set(pd.to_datetime(festivos).normalize())):
        inicio = indice.searchsorted(fecha)
        if inicio < min_train or inicio + horizonte > len(indice):
            continue
        if indice[inicio].normalize() != fecha:      # el festivo debe existir en la serie
            continue
        ventanas.append((inicio, inicio + horizonte, fecha))
    return ventanas[-n_max:]


VENTANAS_F = ventanas_festivos(y.index, FESTIVOS, HORIZONTE)

print(f"Festivos evaluables: {len(VENTANAS_F)}\n")
for a, b, f in VENTANAS_F:
    media = y.iloc[a:b].mean()
    print(f"   {f:%d-%b-%Y} ({f.day_name():<9}) demanda media {media:>8,.0f} MW")

if not VENTANAS_F:
    print("No hay festivos evaluables. Revisa el periodo de datos.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 7.2.2 — Comparación en festivos
# ═══════════════════════════════════════════════════════════════════════════
def _ingenuo_diario(train, h):
    return np.tile(train.iloc[-24:].values, h // 24 + 1)[:h]

def _ingenuo_semanal(train, h):
    return np.tile(train.iloc[-168:].values, h // 168 + 1)[:h]


filas_f, pred_f = [], []

for a, b, fecha in VENTANAS_F:
    ytr, yte = y.iloc[:a], y.iloc[a:b]
    fila = {"fecha": fecha.strftime("%d-%b-%Y"), "dia": fecha.day_name()}

    # Líneas base
    for nombre, p in [("Ingenuo diario",  _ingenuo_diario(ytr, len(yte))),
                      ("Ingenuo semanal", _ingenuo_semanal(ytr, len(yte)))]:
        fila[nombre] = np.mean(np.abs((yte.values - p) / yte.values)) * 100

    # Modelos de aprendizaje automático
    Xtr = df[SEGURAS].iloc[:a]
    Xte = df[SEGURAS].iloc[a:b]
    valido = Xtr.notna().all(axis=1) & ytr.notna()

    for nombre in tabla_ml.index:
        modelo = MODELOS[nombre]()
        modelo.fit(Xtr[valido], ytr[valido])
        p = modelo.predict(Xte)
        fila[nombre] = np.mean(np.abs((yte.values - p) / yte.values)) * 100
        pred_f.append(pd.DataFrame(
            {"real": yte.values, "pred": p, "modelo": nombre,
             "fecha": fecha.strftime("%d-%b")}, index=yte.index))

    filas_f.append(fila)
    print(f"   {fila['fecha']} evaluado")

festivos_tabla = pd.DataFrame(filas_f)
print()
display(festivos_tabla.round(2))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 7.2.3 — Resumen: días ordinarios frente a festivos
# ═══════════════════════════════════════════════════════════════════════════
modelos_cols = [c for c in festivos_tabla.columns if c not in ("fecha", "dia")]

resumen_f = pd.DataFrame({
    "MAPE en festivos": festivos_tabla[modelos_cols].mean(),
    "MAPE peor festivo": festivos_tabla[modelos_cols].max(),
})

# Error en días ordinarios, tomado de la evaluación general
ordinarios = {}
for m in modelos_cols:
    if m in tabla_ml.index:
        ordinarios[m] = tabla_ml.loc[m, "MAPE medio"]
    elif ruta_base.exists() and m in tabla_base.index:
        ordinarios[m] = tabla_base.loc[m, "MAPE medio"]

resumen_f["MAPE ordinario"] = pd.Series(ordinarios)
resumen_f["Penalización"] = resumen_f["MAPE en festivos"] / resumen_f["MAPE ordinario"]
resumen_f = resumen_f.sort_values("MAPE en festivos")

print("Comparación entre días ordinarios y festivos\n")
display(resumen_f.round(2))

print("\nLa penalización indica cuántas veces se multiplica el error en un")
print("festivo respecto a un día ordinario. Un valor próximo a 1 significa que")
print("el modelo aprovecha correctamente la información de calendario.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 7.2.4 — Visualización
# ═══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

orden = resumen_f.index
colores = ["#888780" if m.startswith(("Ingenuo", "Perfil")) else "#4C72B0"
           for m in orden]

x = np.arange(len(orden))
ancho = 0.38
axes[0].bar(x - ancho/2, resumen_f["MAPE ordinario"], ancho,
            label="Día ordinario", color="#55A868")
axes[0].bar(x + ancho/2, resumen_f["MAPE en festivos"], ancho,
            label="Festivo", color="#C44E52")
axes[0].set_xticks(x)
axes[0].set_xticklabels(orden, rotation=20, ha="right", fontsize=8)
axes[0].set_ylabel("MAPE (%)")
axes[0].set_title("Error en días ordinarios frente a festivos", fontweight="bold")
axes[0].legend(fontsize=8)

# Detalle: mejor modelo frente a la línea base en cada festivo
mejor_ml = tabla_ml.index[0]
comp = festivos_tabla.set_index("fecha")[["Ingenuo semanal", mejor_ml]]
comp.plot(kind="barh", ax=axes[1], color=["#888780", "#4C72B0"], width=0.75)
axes[1].set_xlabel("MAPE (%)")
axes[1].set_title("Error festivo a festivo", fontweight="bold")
axes[1].legend(fontsize=8)
axes[1].invert_yaxis()

plt.tight_layout()
guardar_figura("19_evaluacion_festivos")
plt.show()

resumen_f.to_csv(DIR_RESULTADOS / "evaluacion_festivos.csv")
festivos_tabla.to_csv(DIR_RESULTADOS / "festivos_detalle.csv", index=False)
print(f"Guardado en {DIR_RESULTADOS}")

---
## Interpretación para la memoria

Esta evaluación cierra la cadena argumental del trabajo:

1. El análisis de anomalías del notebook 01 identificó que **cerca del 70 % de
   las desviaciones significativas se concentraba en festivos**, con un efecto
   mediano próximo a −7.000 MW.

2. Sobre esa base se incorporaron `festivo`, `vispera_festivo` y `post_festivo`
   al conjunto de predictores.

3. La presente evaluación **verifica que esas variables cumplen su cometido**:
   los modelos que las utilizan mantienen un error contenido en festivos,
   mientras que los modelos ingenuos, que carecen de información de calendario,
   se degradan de forma severa.

El contraste es la mejor demostración del valor del trabajo. La mejora global
del 53 % frente a la línea base es un resultado sólido, pero la diferencia en
festivos es de un orden de magnitud distinto, porque es precisamente donde la
información de calendario resulta determinante.

> **Nota metodológica.** Este conjunto de evaluación está deliberadamente
> sesgado hacia casos difíciles y no es representativo del rendimiento medio
> del sistema. Debe reportarse como análisis complementario, nunca como
> resultado principal. La métrica global sigue siendo la de la sección 5.